# Stage 40 — Classificação com SVM

Objetivo: classificar as quatro palavras separadamente para cada participante. A padronização, seleção de características e SVM ficam no mesmo `Pipeline`.


## 1. Importações


In [ ]:
from pathlib import Path
import pickle
import re

import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


## 2. Caminhos e parâmetros do experimento


In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_30_feature_extraction_psd"
OUTPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_40_svm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_FEATURES = 3
N_SPLITS = 5
RANDOM_STATE = 42
CLASS_LABELS = [0, 1, 2, 3]


## 3. Descobrir os participantes


In [ ]:
feature_files = sorted(INPUT_DIR.glob("sub-*_ses-*_features_psd.npy"))
subjects = sorted({
    re.match(r"(sub-\d+)", path.name).group(1)
    for path in feature_files
})
print("Participantes:", subjects)


## 4. Função para juntar as sessões

As três sessões de um participante são concatenadas no eixo das épocas.


In [ ]:
def load_subject(subject):
    session_files = sorted(INPUT_DIR.glob(f"{subject}_ses-*_features_psd.npy"))
    session_features = []
    session_labels = []

    for feature_path in session_files:
        session_name = feature_path.name.replace("_features_psd.npy", "")
        labels_path = INPUT_DIR / f"{session_name}_labels.npy"
        session_features.append(np.load(feature_path))
        session_labels.append(np.load(labels_path))

    if not session_features:
        raise FileNotFoundError(f"Nenhuma sessão encontrada para {subject}")

    features = np.concatenate(session_features, axis=0)
    labels = np.concatenate(session_labels, axis=0)
    return features, labels


## 5. Inspecionar um participante


In [ ]:
example_features, example_labels = load_subject(subjects[0])
print("Características:", example_features.shape)
print("Rótulos:", example_labels.shape)
print("Classes:", np.unique(example_labels, return_counts=True))


## 6. Construir o pipeline

Em cada fold, o scaler e o seletor são ajustados somente no conjunto de treino.


In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=K_FEATURES)),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale")),
])

cross_validation = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

model


## 7. Função de avaliação de um participante


In [ ]:
def evaluate_subject(subject):
    features, labels = load_subject(subject)
    scores = []
    matrices = []
    selected_features = []

    for train_index, test_index in cross_validation.split(features, labels):
        x_train = features[train_index]
        x_test = features[test_index]
        y_train = labels[train_index]
        y_test = labels[test_index]

        model.fit(x_train, y_train)
        predictions = model.predict(x_test)

        scores.append(accuracy_score(y_test, predictions))
        matrices.append(confusion_matrix(
            y_test,
            predictions,
            labels=CLASS_LABELS,
            normalize="true",
        ))

        indices = np.flatnonzero(model.named_steps["selector"].get_support())
        selected_features.append(indices.tolist())

    scores = np.asarray(scores)
    matrices = np.asarray(matrices)

    return {
        "k_features": K_FEATURES,
        "scores": scores,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std(),
        "confusion_matrices": matrices,
        "mean_confusion_matrix": matrices.mean(axis=0),
        "selected_features": selected_features,
    }


## 8. Avaliar apenas um participante

Use esta célula para entender a saída antes de executar todos.


In [ ]:
example_subject = subjects[0]
example_result = evaluate_subject(example_subject)

print("Participante:", example_subject)
print("Acurácias dos folds:", example_result["scores"])
print("Média:", example_result["mean_accuracy"])
print("Matriz média:\n", example_result["mean_confusion_matrix"])


## 9. Avaliar e salvar todos os participantes


In [ ]:
for subject in subjects:
    result = evaluate_subject(subject)
    output_path = OUTPUT_DIR / f"{subject}_psd_svm_results_k_{K_FEATURES}.pkl"

    with output_path.open("wb") as file:
        pickle.dump({subject: result}, file)

    print(subject, f"acurácia média={result['mean_accuracy']:.3f}")

print("Stage 40 finalizada.")
